In [1]:
from pathlib import Path
from time import perf_counter
import json

import joblib
import pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import ParameterSampler, RandomizedSearchCV, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier

from src.utils.features import split_features

In [2]:
DATA_DIR = Path("../datasets/final/ml")
SEARCH_METHOD = "randomized_search"
MODEL_DIR = Path("../models") / SEARCH_METHOD
RESULTS_DIR = Path("../results/model_training") / SEARCH_METHOD

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_RANDOM_CANDIDATES = 12
N_JOBS = -1

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [3]:
#configurações de treinamento / organização / origem dos dados
TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

In [4]:
# atributos
reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")

numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features

del reference_df

print("features num:", len(numerical_features))
print("features cat:", len(categorical_features))
print("Total de features antes do one-hot:", len(feature_columns))

features num: 14
features cat: 1
Total de features antes do one-hot: 15


In [5]:
#pipeline pré-rocessamento
def build_pipeline(estimator):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"),categorical_features),
        ]
    )

    return Pipeline(
        steps=[ ("preprocessor", preprocessor),
                ("classifier", estimator),
        ]
    )

In [6]:
#modelos e espaços de busca
MODEL_CONFIGS = {
    "decision_tree": {
        "estimator": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "param_distributions": {
            "classifier__max_depth": [None, 3, 5, 8, 10, 15, 20, 30],
            "classifier__min_samples_split": randint(2, 21),
            "classifier__min_samples_leaf": randint(1, 11),
            "classifier__class_weight": [None, "balanced"],
        },
        "baseline_params": {
            "classifier__max_depth": None,
            "classifier__min_samples_split": 2,
            "classifier__min_samples_leaf": 1,
            "classifier__class_weight": None,
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
        ),
        "param_distributions": {
            "classifier__n_estimators": randint(100, 301),
            "classifier__max_depth": [None, 10, 15, 20, 30],
            "classifier__min_samples_leaf": randint(1, 11),
            "classifier__max_features": ["sqrt", "log2", 0.5, 1.0],
            "classifier__class_weight": [None, "balanced", "balanced_subsample"],
        },
        "baseline_params": {
            "classifier__n_estimators": 100,
            "classifier__max_depth": None,
            "classifier__min_samples_leaf": 1,
            "classifier__max_features": "sqrt",
            "classifier__class_weight": None,
        },
    },
    "xgboost": {
        "estimator": XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1,
            eval_metric="logloss",
            tree_method="hist",
        ),
        "param_distributions": {
            "classifier__n_estimators": randint(100, 301),
            "classifier__max_depth": randint(3, 9),
            "classifier__learning_rate": loguniform(0.03, 0.3),
            "classifier__subsample": uniform(0.7, 0.3),
            "classifier__colsample_bytree": uniform(0.7, 0.3),
        },
        # Valores efetivos dos defaults do XGBoost 3.4.1. O wrapper retorna None,
        # mas o booster usa estes valores quando os hiperparâmetros não são informados.
        "baseline_params": {
            "classifier__n_estimators": 100,
            "classifier__max_depth": 6,
            "classifier__learning_rate": 0.3,
            "classifier__subsample": 1.0,
            "classifier__colsample_bytree": 1.0,
            "classifier__eval_metric": "logloss",
            "classifier__tree_method": "hist",
        },
    },
    "linear_svm": {
        "estimator": LinearSVC(
            random_state=RANDOM_STATE,
        ),
        "param_distributions": {
            "classifier__C": loguniform(1e-3, 1e2),
            "classifier__class_weight": [None, "balanced"],
        },
        "baseline_params": {
            "classifier__C": 1.0,
            "classifier__class_weight": None,
            "classifier__dual": "auto",
            "classifier__max_iter": 1000,
        },
    },
    "mlp": {
        "estimator": MLPClassifier(
            random_state=RANDOM_STATE,
        ),
        "param_distributions": {
            "classifier__hidden_layer_sizes": [
                (50,),
                (100,),
                (50, 50),
                (100, 50),
            ],
            "classifier__alpha": loguniform(1e-5, 1e-2),
            "classifier__learning_rate_init": loguniform(1e-4, 1e-2),
            "classifier__early_stopping": [True, False],
        },
        "baseline_params": {
            "classifier__hidden_layer_sizes": (100,),
            "classifier__alpha": 0.0001,
            "classifier__learning_rate_init": 0.001,
            "classifier__early_stopping": False,
            "classifier__max_iter": 200,
        },
    },
}

In [7]:
#modelos para rodar
MODELS_TO_RUN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

In [8]:
# kfold estratificado + métricas
# buscar os melhores hiperparâmetros pelo F1 macro
cross_validation = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

scoring = {
    #o make_scorer cria um avaliador que calcula a métrica estabelecida, considerando a classe 1 ataque como positiva
    "precision_attack": make_scorer(
        precision_score,
        pos_label=1,
        zero_division=0,
    ),
    "recall_attack": make_scorer(
        recall_score,
        pos_label=1,
        zero_division=0,
    ),
    "f1_attack": make_scorer(
        f1_score,
        pos_label=1,
        zero_division=0,
    ),
    "precision_benign": make_scorer(
        precision_score,
        pos_label=0,
        zero_division=0,
    ),
    "recall_benign": make_scorer(
        recall_score,
        pos_label=0,
        zero_division=0,
    ),
    "f1_benign": make_scorer(
        f1_score,
        pos_label=0,
        zero_division=0,
    ),
    "f1_macro": make_scorer(
        f1_score,
        average="macro",
        labels=[0, 1],
        zero_division=0,
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

In [9]:
# carregamento de origem de treinamento
def load_training_data(dataset_names):
    columns = feature_columns + ["label_binary", "dataset_name"]

    dataframes = []

    for dataset_name in dataset_names:
        #columns=columns (pandas pega apenas colunas necessárias)
        df = pd.read_parquet( DATA_DIR / f"{dataset_name}_train.parquet", columns=columns)
        dataframes.append(df)

    training_df = pd.concat( dataframes, ignore_index=True)

    features = training_df[feature_columns]
    # training_df["label_binary"].eq("ATTACK").astype("int8") -> 1 se equal, 0 se não, ou seja, 1 ataque, 0 benigno
    target = (training_df["label_binary"].eq("ATTACK").astype("int8"))
    # Nos treinos combinados, cada fold preserva simultaneamente a origem e a classe.
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    cv_splits = list(cross_validation.split(features, stratification_labels))

    return features, target, cv_splits

In [10]:
##treino, validação e save
def serialize_params(params):
    return json.dumps(
        params,
        sort_keys=True,
        default=lambda value: value.item(),
    )


def resolve_candidate_params(model_config, candidate_params):
    # Registra a configuração completa e reproduzível do estimador.
    effective_params = {
        f"classifier__{name}": value
        for name, value in model_config["estimator"].get_params(deep=False).items()
    }
    effective_params.update(model_config["baseline_params"])
    effective_params.update(candidate_params)
    return effective_params


def build_candidate_grid(model_config):
    # O primeiro candidato é exatamente o baseline, sem sobrescrever parâmetros.
    baseline_fingerprint = serialize_params(
        resolve_candidate_params(model_config, {})
    )
    seen = {baseline_fingerprint}
    random_candidates = []

    # A amostragem excedente permite descartar uma eventual repetição do baseline
    # ou de outro candidato sem reduzir as 12 configurações aleatórias.
    sampled_candidates = ParameterSampler(
        model_config["param_distributions"],
        n_iter=N_RANDOM_CANDIDATES * 20,
        random_state=RANDOM_STATE,
    )

    for candidate_params in sampled_candidates:
        fingerprint = serialize_params(
            resolve_candidate_params(model_config, candidate_params)
        )
        if fingerprint in seen:
            continue

        seen.add(fingerprint)
        random_candidates.append(candidate_params)
        if len(random_candidates) == N_RANDOM_CANDIDATES:
            break

    if len(random_candidates) != N_RANDOM_CANDIDATES:
        raise RuntimeError(
            "Não foi possível gerar 12 candidatos aleatórios únicos."
        )

    # RandomizedSearchCV recebe somente estes candidatos já definidos. Cada valor
    # fica em uma lista unitária para representar uma configuração completa.
    return [{}] + [
        {name: [value] for name, value in params.items()}
        for params in random_candidates
    ]


def train_and_select_model(model_name, training_name, dataset_names):
    features, target, cv_splits = load_training_data(dataset_names)
    model_config = MODEL_CONFIGS[model_name]
    model_dir = MODEL_DIR / model_name
    results_dir = RESULTS_DIR / model_name

    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    candidate_grid = build_candidate_grid(model_config)

    search = RandomizedSearchCV(
        estimator=build_pipeline(model_config["estimator"]),
        param_distributions=candidate_grid,
        n_iter=len(candidate_grid),
        scoring=scoring,
        # O refit treina novamente o pipeline com todo o conjunto de treino,
        # usando os melhores hiperparâmetros encontrados segundo o F1 macro.
        refit="f1_macro",
        cv=cv_splits,
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    start = perf_counter()
    search.fit(features, target)
    elapsed_seconds = perf_counter() - start
    elapsed_minutes = elapsed_seconds / 60

    cv_results = pd.DataFrame(search.cv_results_)

    #informações da busca
    candidate_columns = ["params"]
    for metric in scoring:
        candidate_columns.extend(
            f"split{fold}_test_{metric}" for fold in range(N_SPLITS)
        )
        candidate_columns.extend([
            f"mean_test_{metric}",
            f"std_test_{metric}",
            f"rank_test_{metric}",
        ])
    candidate_columns.extend([
        "mean_fit_time",
        "std_fit_time",
        "mean_score_time",
        "std_score_time",
    ])

    candidate_results = cv_results[candidate_columns].copy()
    candidate_results.insert(0, "candidate_id", range(1, len(candidate_results) + 1))
    candidate_results.insert(
        1,
        "candidate_type",
        candidate_results["params"].map(
            lambda params: "baseline" if not params else "random"
        ),
    )
    candidate_results["params"] = candidate_results["params"].map(
        lambda params: serialize_params(
            resolve_candidate_params(model_config, params)
        )
    )
    candidate_results = candidate_results.sort_values("rank_test_f1_macro")
    candidate_results.insert(0, "random_state", RANDOM_STATE)
    candidate_results.insert(0, "n_splits", N_SPLITS)
    candidate_results.insert(0, "n_candidates", len(cv_results))
    candidate_results.insert(0, "n_random_candidates", N_RANDOM_CANDIDATES)
    candidate_results.insert(0, "training_rows", len(features))
    candidate_results.insert(0, "training_source", training_name)
    candidate_results.insert(0, "search_method", SEARCH_METHOD)
    candidate_results.insert(0, "model", model_name)

    print(f"resultados dos hiperparâmetros: {model_name} - {training_name}")

    #todas as colunas de informações definidas
    display(candidate_results)

    candidate_results.to_csv(
        results_dir / f"{training_name}_candidates.csv",
        index=False,
    )

    model_path = model_dir / f"{training_name}.joblib"
    # Salva o pipeline vencedor, já reajustado com todos os dados de treinamento.
    joblib.dump(
        search.best_estimator_,
        model_path,
        compress=3,
    )

    best_result = cv_results.loc[search.best_index_]
    best_candidate_type = (
        "baseline" if not search.best_params_ else "random"
    )
    best_effective_params = resolve_candidate_params(
        model_config,
        search.best_params_,
    )

    result = {
        "model": model_name,
        "search_method": SEARCH_METHOD,
        "training_source": training_name,
        "training_rows": len(features),
        "n_random_candidates": N_RANDOM_CANDIDATES,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "n_candidates": len(cv_results),
        "best_candidate_type": best_candidate_type,
        "selection_metric": "f1_macro",
        "params": serialize_params(best_effective_params),
        "elapsed_seconds": elapsed_seconds,
        "elapsed_minutes": elapsed_minutes,
        "refit_time_seconds": search.refit_time_,
        "model_size_mb": (
                model_path.stat().st_size / 1024 ** 2
        ),
        "model_file": model_path.name,
    }

    for metric in scoring:
        result[f"{metric}_mean"] = best_result[f"mean_test_{metric}"]
        result[f"{metric}_std"] = best_result[f"std_test_{metric}"]

    return result

In [11]:
# exec
all_results = []

for model_name in MODELS_TO_RUN:
    model_results = []

    for training_name, dataset_names in TRAINING_CONFIGS.items():
        print(f"modelo: {model_name}\n origem do treinamento: {training_name}")

        result = train_and_select_model(
            model_name,
            training_name,
            dataset_names,
        )

        model_results.append(result)
        all_results.append(result)

    results_df = pd.DataFrame(model_results)
    results_df.to_csv(
        RESULTS_DIR / model_name / "summary.csv",
        index=False,
    )

all_results_df = pd.DataFrame(all_results)
display(all_results_df)

modelo: decision_tree
 origem do treinamento: cicids2017
Fitting 5 folds for each of 13 candidates, totalling 65 fits
resultados dos hiperparâmetros: decision_tree - cicids2017


,model,search_method,training_source,training_rows,n_random_candidates,n_candidates,n_splits,random_state,candidate_id,candidate_type,params,split0_test_precision_attack,split1_test_precision_attack,split2_test_precision_attack,split3_test_precision_attack,split4_test_precision_attack,mean_test_precision_attack,std_test_precision_attack,rank_test_precision_attack,split0_test_recall_attack,split1_test_recall_attack,split2_test_recall_attack,split3_test_recall_attack,split4_test_recall_attack,mean_test_recall_attack,std_test_recall_attack,rank_test_recall_attack,split0_test_f1_attack,split1_test_f1_attack,split2_test_f1_attack,split3_test_f1_attack,split4_test_f1_attack,mean_test_f1_attack,std_test_f1_attack,rank_test_f1_attack,split0_test_precision_benign,split1_test_precision_benign,split2_test_precision_benign,split3_test_precision_benign,split4_test_precision_benign,mean_test_precision_benign,std_test_precision_benign,rank_test_precision_benign,split0_test_recall_benign,split1_test_recall_benign,split2_test_recall_benign,split3_test_recall_benign,split4_test_recall_benign,mean_test_recall_benign,std_test_recall_benign,rank_test_recall_benign,split0_test_f1_benign,split1_test_f1_benign,split2_test_f1_benign,split3_test_f1_benign,split4_test_f1_benign,mean_test_f1_benign,std_test_f1_benign,rank_test_f1_benign,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_balanced_accuracy,split1_test_balanced_accuracy,split2_test_balanced_accuracy,split3_test_balanced_accuracy,split4_test_balanced_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,rank_test_balanced_accuracy,mean_fit_time,std_fit_time,mean_score_time,std_score_time
0,decision_tree,randomized_search,cicids2017,235490,12,13,5,1,1,baseline,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 1, ""classifier__min_samples_split"": 2, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.982499,0.982881,0.980702,0.983113,0.983010,0.982441,0.000894,1,0.989383,0.987472,0.987261,0.988854,0.988854,0.988365,0.000840,6,0.985929,0.985171,0.983971,0.985975,0.985923,0.985394,0.000772,1,0.997341,0.996865,0.996810,0.997209,0.997209,0.997087,0.000210,6,0.995594,0.995701,0.995143,0.995753,0.995727,0.995584,0.000227,1,0.996467,0.996282,0.995976,0.996481,0.996467,0.996335,0.000194,1,0.991198,0.990727,0.989973,0.991228,0.991195,0.990864,0.000483,1,0.994352,0.994055,0.993567,0.994373,0.994352,0.994140,0.000310,1,0.992489,0.991586,0.991202,0.992303,0.992290,0.991974,0.000494,6,2.789925,0.088232,0.096048,0.007668
11,decision_tree,randomized_search,cicids2017,235490,12,13,5,1,12,random,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 15, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 17, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.980283,0.982262,0.979716,0.980238,0.981061,0.980712,0.000886,3,0.992356,0.987684,0.989597,0.989915,0.989809,0.989872,0.001486,5,0.986283,0.984966,0.984632,0.985053,0.985415,0.985270,0.000565,2,0.998083,0.996917,0.997392,0.997472,0.997446,0.997462,0.000371,5,0.995010,0.995541,0.994878,0.995010,0.995223,0.995132,0.

modelo: decision_tree
 origem do treinamento: unsw_nb15
Fitting 5 folds for each of 13 candidates, totalling 65 fits
resultados dos hiperparâmetros: decision_tree - unsw_nb15


,model,search_method,training_source,training_rows,n_random_candidates,n_candidates,n_splits,random_state,candidate_id,candidate_type,params,split0_test_precision_attack,split1_test_precision_attack,split2_test_precision_attack,split3_test_precision_attack,split4_test_precision_attack,mean_test_precision_attack,std_test_precision_attack,rank_test_precision_attack,split0_test_recall_attack,split1_test_recall_attack,split2_test_recall_attack,split3_test_recall_attack,split4_test_recall_attack,mean_test_recall_attack,std_test_recall_attack,rank_test_recall_attack,split0_test_f1_attack,split1_test_f1_attack,split2_test_f1_attack,split3_test_f1_attack,split4_test_f1_attack,mean_test_f1_attack,std_test_f1_attack,rank_test_f1_attack,split0_test_precision_benign,split1_test_precision_benign,split2_test_precision_benign,split3_test_precision_benign,split4_test_precision_benign,mean_test_precision_benign,std_test_precision_benign,rank_test_precision_benign,split0_test_recall_benign,split1_test_recall_benign,split2_test_recall_benign,split3_test_recall_benign,split4_test_recall_benign,mean_test_recall_benign,std_test_recall_benign,rank_test_recall_benign,split0_test_f1_benign,split1_test_f1_benign,split2_test_f1_benign,split3_test_f1_benign,split4_test_f1_benign,mean_test_f1_benign,std_test_f1_benign,rank_test_f1_benign,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_balanced_accuracy,split1_test_balanced_accuracy,split2_test_balanced_accuracy,split3_test_balanced_accuracy,split4_test_balanced_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,rank_test_balanced_accuracy,mean_fit_time,std_fit_time,mean_score_time,std_score_time
10,decision_tree,randomized_search,unsw_nb15,235490,12,13,5,1,11,random,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 10, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 10, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.940462,0.944321,0.936937,0.936711,0.941585,0.940003,0.002884,5,0.994479,0.993948,0.993631,0.992994,0.994161,0.993843,0.000507,5,0.966717,0.968499,0.964451,0.964032,0.967159,0.966172,0.001687,1,0.998600,0.998467,0.998383,0.998222,0.998520,0.998438,0.000129,5,0.984262,0.985350,0.983279,0.983226,0.984580,0.984139,0.000806,5,0.991379,0.991865,0.990774,0.990667,0.991501,0.991237,0.000452,1,0.979048,0.980182,0.977612,0.977349,0.979330,0.978704,0.001070,1,0.986305,0.987070,0.985350,0.985180,0.986496,0.986080,0.000714,1,0.989371,0.989649,0.988455,0.988110,0.989371,0.988991,0.000597,5,2.667162,0.076821,0.093703,0.015051
3,decision_tree,randomized_search,unsw_nb15,235490,12,13,5,1,4,random,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": ""balanced"", ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 10, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 8, ""classifier__min_samples_split"": 15, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.936280,0.940118,0.933062,0.933287,0.936809,0.935911,0.002594,11,0.998407,0.998407,0.998832,0.997983,0.997771,0.998280,0.000370,1,0.966346,0.968386,0.964828,0.964551,0.966329,0.966088,0.001368,2,0.999595,0.999596,0.999703,0.999487,0.999433,0.999563,0.000094,1,0.983014,0.984103,0.982085,0.982165,0.983173,0.98290

modelo: decision_tree
 origem do treinamento: iot23
Fitting 5 folds for each of 13 candidates, totalling 65 fits
resultados dos hiperparâmetros: decision_tree - iot23


,model,search_method,training_source,training_rows,n_random_candidates,n_candidates,n_splits,random_state,candidate_id,candidate_type,params,split0_test_precision_attack,split1_test_precision_attack,split2_test_precision_attack,split3_test_precision_attack,split4_test_precision_attack,mean_test_precision_attack,std_test_precision_attack,rank_test_precision_attack,split0_test_recall_attack,split1_test_recall_attack,split2_test_recall_attack,split3_test_recall_attack,split4_test_recall_attack,mean_test_recall_attack,std_test_recall_attack,rank_test_recall_attack,split0_test_f1_attack,split1_test_f1_attack,split2_test_f1_attack,split3_test_f1_attack,split4_test_f1_attack,mean_test_f1_attack,std_test_f1_attack,rank_test_f1_attack,split0_test_precision_benign,split1_test_precision_benign,split2_test_precision_benign,split3_test_precision_benign,split4_test_precision_benign,mean_test_precision_benign,std_test_precision_benign,rank_test_precision_benign,split0_test_recall_benign,split1_test_recall_benign,split2_test_recall_benign,split3_test_recall_benign,split4_test_recall_benign,mean_test_recall_benign,std_test_recall_benign,rank_test_recall_benign,split0_test_f1_benign,split1_test_f1_benign,split2_test_f1_benign,split3_test_f1_benign,split4_test_f1_benign,mean_test_f1_benign,std_test_f1_benign,rank_test_f1_benign,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_balanced_accuracy,split1_test_balanced_accuracy,split2_test_balanced_accuracy,split3_test_balanced_accuracy,split4_test_balanced_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,rank_test_balanced_accuracy,mean_fit_time,std_fit_time,mean_score_time,std_score_time
4,decision_tree,randomized_search,iot23,235490,12,13,5,1,5,random,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 20, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 10, ""classifier__min_samples_split"": 20, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.999680,0.999893,0.998507,0.999680,0.999466,0.999445,0.000488,1,0.994373,0.993524,0.994161,0.993737,0.993418,0.993843,0.000368,11,0.997019,0.996698,0.996330,0.996699,0.996433,0.996636,0.000241,1,0.998595,0.998384,0.998542,0.998436,0.998357,0.998463,0.000092,11,0.999920,0.999973,0.999628,0.999920,0.999867,0.999862,0.000122,1,0.999257,0.999178,0.999085,0.999178,0.999112,0.999162,0.000060,1,0.998138,0.997938,0.997707,0.997939,0.997772,0.997899,0.000150,1,0.998811,0.998684,0.998535,0.998684,0.998577,0.998658,0.000096,1,0.997147,0.996749,0.996895,0.996829,0.996643,0.996852,0.000170,8,1.603006,0.213707,0.096732,0.009410
10,decision_tree,randomized_search,iot23,235490,12,13,5,1,11,random,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 10, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 10, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",0.998614,0.999680,0.999146,0.999466,0.999680,0.999317,0.000402,3,0.994161,0.993948,0.994161,0.993843,0.993418,0.993906,0.000273,8,0.996382,0.996806,0.996648,0.996647,0.996539,0.996604,0.000140,2,0.998542,0.998489,0.998542,0.998463,0.998357,0.998479,0.000068,8,0.999655,0.999920,0.999788,0.999867,0.999920,0.999830,0.000100,3,0

... (51 outputs left out)

,model,search_method,training_source,training_rows,n_random_candidates,n_candidates,n_splits,random_state,candidate_id,candidate_type,params,split0_test_precision_attack,split1_test_precision_attack,split2_test_precision_attack,split3_test_precision_attack,split4_test_precision_attack,mean_test_precision_attack,std_test_precision_attack,rank_test_precision_attack,split0_test_recall_attack,split1_test_recall_attack,split2_test_recall_attack,split3_test_recall_attack,split4_test_recall_attack,mean_test_recall_attack,std_test_recall_attack,rank_test_recall_attack,split0_test_f1_attack,split1_test_f1_attack,split2_test_f1_attack,split3_test_f1_attack,split4_test_f1_attack,mean_test_f1_attack,std_test_f1_attack,rank_test_f1_attack,split0_test_precision_benign,split1_test_precision_benign,split2_test_precision_benign,split3_test_precision_benign,split4_test_precision_benign,mean_test_precision_benign,std_test_precision_benign,rank_test_precision_benign,split0_test_recall_benign,split1_test_recall_benign,split2_test_recall_benign,split3_test_recall_benign,split4_test_recall_benign,mean_test_recall_benign,std_test_recall_benign,rank_test_recall_benign,split0_test_f1_benign,split1_test_f1_benign,split2_test_f1_benign,split3_test_f1_benign,split4_test_f1_benign,mean_test_f1_benign,std_test_f1_benign,rank_test_f1_benign,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_balanced_accuracy,split1_test_balanced_accuracy,split2_test_balanced_accuracy,split3_test_balanced_accuracy,split4_test_balanced_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,rank_test_balanced_accuracy,mean_fit_time,std_fit_time,mean_score_time,std_score_time
6,mlp,randomized_search,cicids2017_iot23,470980,12,13,5,1,7,random,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 0.001026603003516424, ""classifier__batch_size"": ""auto"", ""classifier__beta_1"": 0.9, ""classifier__beta_2"": 0.999, ""classifier__early_stopping"": false, ""classifier__epsilon"": 1e-08, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate"": ""constant"", ""classifier__learning_rate_init"": 0.0013103269001464436, ""classifier__max_fun"": 15000, ""classifier__max_iter"": 200, ""classifier__momentum"": 0.9, ""classifier__n_iter_no_change"": 10, ""classifier__nesterovs_momentum"": true, ""classifier__power_t"": 0.5, ""classifier__random_state"": 1, ""classifier__shuffle"": true, ""classifier__solver"": ""adam"", ""classifier__tol"": 0.0001, ""classifier__validation_fraction"": 0.1, ""classifier__verbose"": false, ""classifier__warm_start"": false}",0.987153,0.982325,0.987195,0.986749,0.987422,0.986169,0.001934,3,0.991188,0.994267,0.990287,0.988110,0.987527,0.990276,0.002409,1,0.989166,0.988260,0.988738,0.987429,0.987474,0.988214,0.000685,1,0.997795,0.998562,0.997570,0.997026,0.996881,0.997567,0.000601,1,0.996775,0.995528,0.996789,0.996682,0.996855,0.996526,0.000502,3,0.997285,0.997043,0.997179,0.996854,0.996868,0.997046,0.000169,1,0.993226,0.992652,0.992959,0.992142,0.992171,0.992630,0.000427,1,0.995658,0.995276,0.995488,0.994968,0.994989,0.995276,0.000271,1,0.993982,0.994897,0.993538,0.992396,0.992191,0.993401,0.001007,1,151.148397,9.186442,0.251825,0.019729
7,mlp,randomized_search,cicids2017_iot23,470980,12,13,5,1,8,random,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 2.6373077790924614e-05, ""classifier__batch_size"": ""auto"", ""classifier__beta_1"": 0.9, ""classifier__beta_2"": 0.999, ""classifier__early_stopping"": true, ""classifier__epsilon"": 1e-08, ""classifier__hidden_layer_sizes"": [50, 50], ""classifier__learning_rate"": ""constant"", ""classifier__learning_rate_init"": 0.003994745694560914, ""classifier__max_fun"": 15000, ""classifier__max

modelo: mlp
 origem do treinamento: unsw_nb15_iot23
Fitting 5 folds for each of 13 candidates, totalling 65 fits
resultados dos hiperparâmetros: mlp - unsw_nb15_iot23


,model,search_method,training_source,training_rows,n_random_candidates,n_candidates,n_splits,random_state,candidate_id,candidate_type,params,split0_test_precision_attack,split1_test_precision_attack,split2_test_precision_attack,split3_test_precision_attack,split4_test_precision_attack,mean_test_precision_attack,std_test_precision_attack,rank_test_precision_attack,split0_test_recall_attack,split1_test_recall_attack,split2_test_recall_attack,split3_test_recall_attack,split4_test_recall_attack,mean_test_recall_attack,std_test_recall_attack,rank_test_recall_attack,split0_test_f1_attack,split1_test_f1_attack,split2_test_f1_attack,split3_test_f1_attack,split4_test_f1_attack,mean_test_f1_attack,std_test_f1_attack,rank_test_f1_attack,split0_test_precision_benign,split1_test_precision_benign,split2_test_precision_benign,split3_test_precision_benign,split4_test_precision_benign,mean_test_precision_benign,std_test_precision_benign,rank_test_precision_benign,split0_test_recall_benign,split1_test_recall_benign,split2_test_recall_benign,split3_test_recall_benign,split4_test_recall_benign,mean_test_recall_benign,std_test_recall_benign,rank_test_recall_benign,split0_test_f1_benign,split1_test_f1_benign,split2_test_f1_benign,split3_test_f1_benign,split4_test_f1_benign,mean_test_f1_benign,std_test_f1_benign,rank_test_f1_benign,split0_test_f1_macro,split1_test_f1_macro,split2_test_f1_macro,split3_test_f1_macro,split4_test_f1_macro,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro,split0_test_accuracy,split1_test_accuracy,split2_test_accuracy,split3_test_accuracy,split4_test_accuracy,mean_test_accuracy,std_test_accuracy,rank_test_accuracy,split0_test_balanced_accuracy,split1_test_balanced_accuracy,split2_test_balanced_accuracy,split3_test_balanced_accuracy,split4_test_balanced_accuracy,mean_test_balanced_accuracy,std_test_balanced_accuracy,rank_test_balanced_accuracy,mean_fit_time,std_fit_time,mean_score_time,std_score_time
6,mlp,randomized_search,unsw_nb15_iot23,470980,12,13,5,1,7,random,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 0.001026603003516424, ""classifier__batch_size"": ""auto"", ""classifier__beta_1"": 0.9, ""classifier__beta_2"": 0.999, ""classifier__early_stopping"": false, ""classifier__epsilon"": 1e-08, ""classifier__hidden_layer_sizes"": [100, 50], ""classifier__learning_rate"": ""constant"", ""classifier__learning_rate_init"": 0.0013103269001464436, ""classifier__max_fun"": 15000, ""classifier__max_iter"": 200, ""classifier__momentum"": 0.9, ""classifier__n_iter_no_change"": 10, ""classifier__nesterovs_momentum"": true, ""classifier__power_t"": 0.5, ""classifier__random_state"": 1, ""classifier__shuffle"": true, ""classifier__solver"": ""adam"", ""classifier__tol"": 0.0001, ""classifier__validation_fraction"": 0.1, ""classifier__verbose"": false, ""classifier__warm_start"": false}",0.966772,0.967787,0.966838,0.963835,0.967315,0.966509,0.001386,1,0.991560,0.984022,0.988854,0.993047,0.992781,0.990053,0.003361,3,0.979009,0.975837,0.977722,0.978223,0.979883,0.978135,0.001362,1,0.997876,0.995989,0.997197,0.998248,0.998183,0.997499,0.000842,3,0.991481,0.991812,0.991520,0.990684,0.991613,0.991422,0.000386,1,0.994668,0.993896,0.994351,0.994452,0.994887,0.994451,0.000333,1,0.986839,0.984867,0.986036,0.986337,0.987385,0.986293,0.000848,1,0.991496,0.990254,0.990987,0.991157,0.991847,0.991148,0.000536,1,0.991520,0.987917,0.990187,0.991865,0.992197,0.990737,0.001567,2,217.051300,17.394591,0.262855,0.017377
7,mlp,randomized_search,unsw_nb15_iot23,470980,12,13,5,1,8,random,"{""classifier__activation"": ""relu"", ""classifier__alpha"": 2.6373077790924614e-05, ""classifier__batch_size"": ""auto"", ""classifier__beta_1"": 0.9, ""classifier__beta_2"": 0.999, ""classifier__early_stopping"": true, ""classifier__epsilon"": 1e-08, ""classifier__hidden_layer_sizes"": [50, 50], ""classifier__learning_rate"": ""constant"", ""classifier__learning_rate_init"": 0.003994745694560914, ""classifier__max_fun"": 15000, ""classifier__max_

,model,search_method,training_source,training_rows,n_random_candidates,n_splits,random_state,n_candidates,best_candidate_type,selection_metric,params,elapsed_seconds,elapsed_minutes,refit_time_seconds,model_size_mb,model_file,precision_attack_mean,precision_attack_std,recall_attack_mean,recall_attack_std,f1_attack_mean,f1_attack_std,precision_benign_mean,precision_benign_std,recall_benign_mean,recall_benign_std,f1_benign_mean,f1_benign_std,f1_macro_mean,f1_macro_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,decision_tree,randomized_search,cicids2017,235490,12,5,1,13,baseline,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 1, ""classifier__min_samples_split"": 2, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",26.331770,0.438863,1.509994,0.032135,cicids2017.joblib,0.982441,0.000894,0.988365,0.000840,0.985394,0.000772,0.997087,0.000210,0.995584,0.000227,0.996335,0.000194,0.990864,0.000483,0.994140,0.000310,0.991974,0.000494
1,decision_tree,randomized_search,unsw_nb15,235490,12,5,1,13,random,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 10, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 2, ""classifier__min_samples_split"": 10, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",17.191190,0.286520,1.351823,0.014352,unsw_nb15.joblib,0.940003,0.002884,0.993843,0.000507,0.966172,0.001687,0.998438,0.000129,0.984139,0.000806,0.991237,0.000452,0.978704,0.001070,0.986080,0.000714,0.988991,0.000597
2,decision_tree,randomized_search,iot23,235490,12,5,1,13,random,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 20, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 10, ""classifier__min_samples_split"": 20, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",9.599222,0.159987,0.470906,0.007753,iot23.joblib,0.999445,0.000488,0.993843,0.000368,0.996636,0.000241,0.998463,0.000092,0.999862,0.000122,0.999162,0.000060,0.997899,0.000150,0.998658,0.000096,0.996852,0.000170
3,decision_tree,randomized_search,cicids2017_unsw_nb15,470980,12,5,1,13,random,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": 15, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impurity_decrease"": 0.0, ""classifier__min_samples_leaf"": 4, ""classifier__min_samples_split"": 17, ""classifier__min_weight_fraction_leaf"": 0.0, ""classifier__monotonic_cst"": null, ""classifier__random_state"": 1, ""classifier__splitter"": ""best""}",41.793368,0.696556,3.751777,0.037890,cicids2017_unsw_nb15.joblib,0.959575,0.000273,0.987059,0.001120,0.973122,0.000515,0.996741,0.000281,0.989604,0.000078,0.993160,0.000126,0.983141,0.000321,0.989095,0.000203,0.988332,0.000543
4,decision_tree,randomized_search,cicids2017_iot23,470980,12,5,1,13,baseline,f1_macro,"{""classifier__ccp_alpha"": 0.0, ""classifier__class_weight"": null, ""classifier__criterion"": ""gini"", ""classifier__max_depth"": null, ""classifier__max_features"": null, ""classifier__max_leaf_nodes"": null, ""classifier__min_impu